# Lab 1: Agent CLI Single-Turn com Tool-use

## Setup
Importando as dependências necessárias e inicializando o cliente OpenAI. O cliente usará o endpoint da Nvidia e carregará sua chave a partir do `.env`.

In [1]:
import json
import os
from typing import Any
from openai import OpenAI
from dotenv import load_dotenv

# Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

# Inicializa o cliente OpenAI usando a API da Nvidia
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ.get("NVIDIA_API_KEY")
)
MODEL = "qwen/qwen3-coder-480b-a35b-instruct"

## Etapa 1. Definir duas tools com schema JSON
Aqui definimos duas ferramentas (tools) locais: uma calculadora (`calculator`) segura usando `eval`, e uma busca de documentação (`lookup_doc`).
Também criamos o JSON schema dessas funções na variável `TOOLS` e as mapeamos no `TOOL_REGISTRY`.

In [2]:
def calculator(expression: str) -> str:
    """Avalia expressao aritmetica simples."""
    allowed = set("0123456789+-*/(). ")
    if not all(c in allowed for c in expression):
        return "ERROR: expressao contem caracteres nao permitidos"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"ERROR: {e}"

DOCS = {
    "retry": "Use exponential backoff entre tentativas para evitar thundering herd.",
    "pydantic": "Pydantic valida payload via BaseModel + type hints.",
    "streaming": "Streaming reduz time-to-first-token mas dificulta retry idempotente.",
}

def lookup_doc(term: str) -> str:
    """Consulta documentacao local por termo exato (case-insensitive)."""
    return DOCS.get(term.lower(), f"NOT_FOUND: termo '{term}' nao encontrado")

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Avalia uma expressao aritmetica simples (apenas + - * / e parenteses).",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Expressao matematica, ex: '12 * (3 + 4)'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_doc",
            "description": "Consulta um termo na base de documentacao local (retry, pydantic, streaming).",
            "parameters": {
                "type": "object",
                "properties": {
                    "term": {
                        "type": "string",
                        "description": "Termo a buscar, ex: 'retry'",
                    }
                },
                "required": ["term"],
            },
        },
    },
]

TOOL_REGISTRY = {"calculator": calculator, "lookup_doc": lookup_doc}

## Etapa 2. Loop tool, LLM, tool (Single-Turn Agent)
Nesta etapa criamos a função `run_agent` que orquestra as interações. O LLM recebe as `TOOLS`. Caso ele decida chamar uma ferramenta (`msg.tool_calls`), nós a executamos localmente, adicionamos o resultado no histórico e chamamos o LLM novamente. Esse loop continua até que o LLM dê a resposta final.

In [3]:
def run_agent(user_query: str, max_iters: int = 5) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "Voce e um assistente que pode usar tools. "
                "Use 'calculator' para contas e 'lookup_doc' para definicoes tecnicas. "
                "Sempre cite a fonte quando usar lookup_doc. "
                "Se uma tool retornar ERROR, NAO repita a mesma chamada; resuma o erro ao usuario."
            ),
        },
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iters):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        
        # Adiciona a resposta (que pode conter as chamadas de tool) ao histórico
        messages.append(msg.model_dump(exclude_unset=True))
        
        # Se não houver chamadas de ferramenta, temos a resposta final
        if not msg.tool_calls:
            return msg.content or ""
        
        # Processando cada chamada de ferramenta solicitada pelo LLM
        for call in msg.tool_calls:
            fn_name = call.function.name
            
            try:
                args = json.loads(call.function.arguments)
                result = TOOL_REGISTRY[fn_name](**args)
            except json.JSONDecodeError:
                result = "ERROR: invalid arguments"
            
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": str(result),
                }
            )
            
    return "[ERROR] max iterations excedidas"

## Ponto de Verificação
Vamos testar nosso agente para garantir que as ferramentas estão sendo chamadas corretamente.
A terceira query testará múltiplas ferramentas na mesma interação (conversa aberta).

In [4]:
print("--- Teste 1: Calculadora ---")
print(run_agent("Quanto e 47 * 13 + 200?"))
print("\n--- Teste 2: Documentação ---")
print(run_agent("O que e retry em chamadas HTTP?"))
print("\n--- Teste 3: Múltiplas ferramentas (Calculadora + Documentação) ---")
print(run_agent("Calcule 25% de 480 e me explique o que e pydantic"))

--- Teste 1: Calculadora ---
O resultado de 47 * 13 + 200 é 811.

--- Teste 2: Documentação ---
"Retry" em chamadas HTTP refere-se à prática de tentar novamente uma requisição que falhou, geralmente devido a erros temporários, como timeouts ou erros de rede. A abordagem de "retry" é importante para melhorar a resiliência de uma aplicação ao lidar com serviços externos.

Um conceito associado é o uso de "exponential backoff" entre as tentativas. Isso significa que, em vez de repetir a requisição imediatamente após uma falha, o tempo de espera entre uma tentativa e outra aumenta exponencialmente. Isso ajuda a evitar o problema conhecido como "thundering herd", no qual muitas requisições simultâneas sobrecarregam o servidor após uma falha, dificultando sua recuperação.

Portanto, usar "retry" com "exponential backoff" é uma boa prática para manter a estabilidade e a performance de sistemas distribuídos.

--- Teste 3: Múltiplas ferramentas (Calculadora + Documentação) ---
25% de 480 é 120.

## Etapa 3. Comparativo entre pure-prompt e tool-use
Agora faremos a mesma requisição sem passar a lista de ferramentas (`TOOLS`). O modelo será forçado a responder usando apenas seu conhecimento interno (Pure Prompt).
Isso nos permite comparar em quais casos usar ferramentas é indispensável (ex: cálculos precisos ou dados em tempo real) e onde é excesso de engenharia (overkill).

In [5]:
def run_pure_prompt(user_query: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Responda diretamente. Faca contas mentalmente."},
            {"role": "user", "content": user_query},
        ],
    )
    return response.choices[0].message.content or ""

print("--- Teste 1 (Pure Prompt) ---")
print(run_pure_prompt("Quanto e 47 * 13 + 200?"))
print("\n--- Teste 2 (Pure Prompt) ---")
print(run_pure_prompt("O que e retry em chamadas HTTP?"))
print("\n--- Teste 3 (Pure Prompt) ---")
print(run_pure_prompt("Calcule 25% de 480 e me explique o que e pydantic"))

--- Teste 1 (Pure Prompt) ---
Vou calcular mentalmente:

47 × 13 + 200

Primeiro calculo 47 × 13:
- 47 × 10 = 470
- 47 × 3 = 141
- 470 + 141 = 611

Agora somo 200:
611 + 200 = 811

Resposta: 811

--- Teste 2 (Pure Prompt) ---
"Retry" em chamadas HTTP significa **tentar novamente** uma requisição que falhou.

Quando uma chamada HTTP falha (ex: timeout, erro de rede, ou código de erro 5xx), o "retry" é o processo de repetir a mesma requisição, geralmente após um intervalo de tempo, para tentar obter sucesso.

### Exemplo:
- Você faz um `POST /api/users`
- O servidor responde com `500 Internal Server Error`
- Com retry, você tenta novamente a mesma requisição

### Estratégias comuns:
- **Backoff exponencial**: aumentar o tempo entre tentativas (1s, 2s, 4s…)
- **Número máximo de tentativas**: ex: tentar no máximo 3 vezes
- **Timeouts e jitter**: evita sobrecarga do servidor

### Quando usar:
- Erros temporários (503, 504, timeouts)
- Problemas de rede passageiros
- APIs com limitação de ta

## Resumo: Tabela Comparativa

| Query | Pure-prompt | Tool-use | Recomendado |
| :--- | :--- | :--- | :--- |
| **Aritmética com mais de 2 dígitos** | Erro silencioso ocasional | Sempre exato | **Tool-use** |
| **Definição factual** *(ex: Regra interna)* | Depende do *training cutoff* | Ancorado em fonte da verdade | **Tool-use** |
| **Conversa aberta** *(ex: Conceitos gerais)* | Natural, fluida e imediata | *Overhead* desnecessário | **Pure-prompt** |